# Cleansing: เตรียมไฟล์ CIC-IDS2017

Notebook นี้มีโค้ดทำงานจริงอยู่ใน cells ด้านล่าง ตามรูปแบบของ NSL-KDD และ UNSW-NB15 ไม่มีการเรียก `run_pipeline.py` หรือฟังก์ชันประมวลผลจากไฟล์ `.py` แยก

งานในฉบับนี้ตรงกับ implementation เดิม: อ่าน raw CSV, ปรับ BOM/ช่องว่างในชื่อคอลัมน์, เพิ่ม `orig_row_id`, ทำเครื่องหมายแถวซ้ำใน `_duplicate_flag` และบันทึก audit/manifest โดย **ไม่ลบแถวและไม่เติม missing values**

ข้อมูลใน working copy ที่ได้จาก `download.py` เป็นข้อมูลตัวอย่างสังเคราะห์ ไม่ใช่ raw benchmark ทางการ; ตรวจ provenance ก่อนอ้างผล


## 1. กำหนด path และตรวจไฟล์ต้นทาง

ค้นหาโฟลเดอร์จากตำแหน่งที่เปิด notebook ไม่ผูกกับ path ของเครื่องใดเครื่องหนึ่ง จากนั้นยืนยันว่ามี raw train/test ครบก่อนเริ่ม.

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display

DATASET_NAME = 'CIC-IDS2017'
RAW_FILES = {"train": 'CIC_IDS2017_training-set.csv', "test": 'CIC_IDS2017_testing-set.csv'}

def find_dataset_dir(start: Path, dataset_name: str) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if candidate.name == dataset_name and (candidate / "download.py").is_file():
            return candidate
        nested = candidate / "data-pipeline" / "sub-dataset" / dataset_name
        if (nested / "download.py").is_file():
            return nested
    raise FileNotFoundError(f"หาโฟลเดอร์ {dataset_name} จาก {start} ไม่พบ")

DATASET_DIR = find_dataset_dir(Path.cwd(), DATASET_NAME)
RAW_DIR = DATASET_DIR / "raw"
OUTPUT_DIR = DATASET_DIR / "output"
CLEAN_DIR = OUTPUT_DIR / "cleansing"
HARM_DIR = OUTPUT_DIR / "harmonized"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

missing_raw = [name for name in RAW_FILES.values() if not (RAW_DIR / name).is_file()]
if missing_raw:
    raise FileNotFoundError(f"ไม่พบ raw files {missing_raw} ใน {RAW_DIR}; เตรียม raw ก่อน")

print("Dataset:", DATASET_NAME)
print("Raw:", RAW_DIR)
print("Cleansing output:", CLEAN_DIR)


## 2. อ่าน `raw CSV` และทำเครื่องหมายโดยไม่ทิ้งแถว

`orig_row_id` ช่วยย้อนกลับไปยังตำแหน่งแถวในไฟล์ ส่วน `_duplicate_flag` ใช้บอกแถวซ้ำที่ตรวจพบ.

In [ ]:
audit_rows = []
manifest_rows = []
cleaned_columns = {}

for split_name, filename in RAW_FILES.items():
    source_path = RAW_DIR / filename
    output_path = CLEAN_DIR / filename
    print(f"Reading {split_name}: {source_path.name}")

    frame = pd.read_csv(source_path, low_memory=False)
    rows_in = len(frame)

    # ทำมาตรฐานชื่อ header และคงแถวไว้ พร้อมติดธง duplicate เพื่อให้ตรวจย้อนหลังได้
    frame.columns = [str(column).replace("\ufeff", "").strip() for column in frame.columns]
    for column in frame.select_dtypes(include=["object", "string"]).columns:
        frame[column] = frame[column].astype("string").str.strip()
    duplicate_flag = frame.duplicated().astype(int)
    if "orig_row_id" not in frame.columns:
        frame["orig_row_id"] = np.arange(len(frame), dtype=np.int64)
    frame["_duplicate_flag"] = duplicate_flag

    frame.to_csv(output_path, index=False)
    cleaned_columns[split_name] = list(frame.columns)
    audit_rows.append({
        "file": filename,
        "missing_cells": int(frame.isna().sum().sum()),
        "output_file": filename,
        "rows_in": rows_in,
        "rows_out": len(frame),
        "rows_removed": 0,
        "changed_cells": 0,
        "rows_with_changed_values": 0,
        "trimmed_cells": 0,
        "sentinel_to_missing_cells": 0,
        "missing_cells_before": int(frame.isna().sum().sum()),
        "missing_cells_after": int(frame.isna().sum().sum()),
        "duplicate_rows_flagged": int(frame["_duplicate_flag"].sum()),
        "orig_row_ids_added": len(frame),
        "data_columns_added": "orig_row_id, _duplicate_flag",
    })
    manifest_rows.append({
        "file": filename,
        "bytes": source_path.stat().st_size,
        "sha256": sha256_file(source_path),
    })

audit = pd.DataFrame(audit_rows)
manifest = pd.DataFrame(manifest_rows)


## 3. บันทึกผล cleansing และตรวจ audit

ไฟล์ cleaned อยู่ใน `output/cleansing/`; manifest ใน implementation นี้เก็บ hash ของไฟล์ cleaned ที่สร้าง.

In [ ]:
audit_path = CLEAN_DIR / "cleansing_audit.csv"
manifest_path = CLEAN_DIR / "source_manifest.csv"
audit.to_csv(audit_path, index=False)
manifest.to_csv(manifest_path, index=False)

print("Cleansing summary (this implementation flags duplicates; it does not drop rows or impute missing values):")
display(audit)
print("Cleaned columns:")
display(pd.DataFrame({name: columns for name, columns in cleaned_columns.items()}))
print("Saved:", audit_path)
print("Saved:", manifest_path)


## ผลลัพธ์

- ไฟล์ cleaned แยก train/test
- `cleansing_audit.csv`
- `source_manifest.csv`

ขั้นถัดไปคือ `harmonization.ipynb` ซึ่งอ่านไฟล์ cleaned เหล่านี้แล้วสร้าง `features`, `labels`, `metadata`.